In [42]:
import numpy as np
import json
from classy import Class
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import cosmoprimo
import os

from FishLSS.fisherForecast import fisherForecast
from FishLSS.experiment import experiment

In [10]:
# import scipy
# print(scipy.__version__)

In [11]:
# import numpy as np
# print(np.__version__)

In [12]:
# filename for example survey
bfn = 'DESI2_fullsky'
# bfn = 'example_survey'
bd = '/home/adrien/PDM/code/PDM2026_wsl/derivatives/'

In [13]:
# path = '/home/adrien/PDM/code/FishLSS/notebooks/DESI.txt'

# zedges = []
# data = np.loadtxt(path, unpack=True)
# redshifts = data[0]
# memory = 0
# for i in range(len(redshifts)):
#     if i == 0:
#         a = redshifts[i+1] - redshifts[i]
#         zedges.append(redshifts[i] - a/2)
#         zedges.append(redshifts[i] + a/2)
#         memory = a
#     elif i < len(redshifts)-1:
#         a = redshifts[i] - redshifts[i-1]
#         b = redshifts[i+1] - redshifts[i]
#         c = min(a, b)
#         zedges.append(redshifts[i] - c/2)
#         zedges.append(redshifts[i] + c/2)
#     else:
#         a = redshifts[i] - redshifts[i-1]
#         zedges.append(redshifts[i] - a/2)
#         zedges.append(redshifts[i] + a/2)


# print(zedges)

## Steps
- define survey (create example_survey.txt with three columns: redshift, $b(z)$, $n(z)$)
- open `example_setup_BAO_sigma8.py` and set `bd`, zmin, zmax, nbins, fsky. Or give nbins and fsky as parameters when runing.

## For DESI
- $n(z)$ can be seen in Fig4 of 2503.14738
- $b(z)$ scaling approximation
- $f_\mathrm{sky}$ --> BGS 12355 $\mathrm{deg}^2$, LRG 10031 $\mathrm{deg}^2$, ELG 10352 $\mathrm{deg}^2$, QSO 11182 $\mathrm{deg}^2$. The full sky : $4 \pi 180^2 / \pi^2 \approx 41253$ $\mathrm{deg}^2$. Most conservative is LRG with $f_\mathrm{sky} \sim 0.243$
- zbins should be given but we'll see that later for now well take the linspaced bins

In [14]:
# sky = 4 * 180**2 / np.pi
# f_BGS = 12355/sky
# f_LRG = 10031/sky
# f_ELG = 10352/sky
# f_QSO = 11182/sky
# print(f_BGS, f_LRG, f_ELG, f_QSO)

-------

##  <span style="color:red"> (0) To speed up the process: </span>

It is significantly faster to set up the forecast (and compute derivatives) with a dedicated computing node rather than from within a jupyter notebook. To do so, run the following from an interactive node (from within the `FishLSS` directory). 

Change `bd` in `example_setup_BAO_sigma8.py` before running this. The derivatives will be saved in `bd+'ouput/'+bfn`

Run :
```
python example_setup_BAO_sigma8.py example_survey setup &
python example_setup_BAO_sigma8.py example_survey rec &
python example_setup_BAO_sigma8.py example_survey fs &
python example_setup_BAO_sigma8.py example_survey lens &
wait
```

and replace `example survey` with `bfn`. If any of these process run over the time limit, simply rerun them. $\verb|FishLSS|$ will pick up where it left off.

Optional: the number of bins and $f_\text{sky}$ can be passed as arguments when calling `example_setup_BAO_sigma8.py`, should you want to change them from their default values (3, 0.5 respectively for this example).

-------

```
python example_setup_BAO_sigma8.py DESI2_fullsky setup 7 0.75 &
python example_setup_BAO_sigma8.py DESI2_fullsky rec 7 0.75 &
python example_setup_BAO_sigma8.py DESI2_fullsky fs 7 0.75 &
python example_setup_BAO_sigma8.py DESI2_fullsky lens 7 0.75 &
wait
```

In [15]:
log = np.load('/home/adrien/miniconda3/envs/fishlss/lib/python3.11/site-packages/FishLSS/bao_recon/log.npy', allow_pickle=True).item()

In [16]:
for key in log.keys():
    print(key, log[key])

n_tot 84


## (1) Setting up a $\verb|FishLSS|$ forecast

Most of the work has already been done in step <span style="color:red"> (0) </span>. After setting up a forecast, $\verb|FishLSS|$ will create a JSON file that summarizes the forecast assumptions and inputs. We will use this to load the relevant information when building the forecast from within the notebook.

In [17]:
f = open('../derivatives/output/'+bfn+'/summary.json')
summary = json.load(f)

In [18]:
print('summary = ')
for keys in summary.keys():
    print(keys, summary[keys])

summary = 
Forecast name DESI2_fullsky
Edges of redshift bins [0.15, 0.4714285714285714, 0.7928571428571429, 1.1142857142857143, 1.4357142857142857, 1.7571428571428571, 2.0785714285714287, 2.4]
Centers of redshift bins [0.3107142857142857, 0.6321428571428571, 0.9535714285714286, 1.275, 1.5964285714285715, 1.917857142857143, 2.2392857142857143]
Linear Eulerian bias in each bin [1.65, 2.0, 1.8928571428571428, 1.2500000000000002, 1.5952380952380953, 2.0, 2.0]
Number density in each bin [0.0008, 0.0005, 0.0002866071428571428, 0.00020625000000000005, 0.00011107142857142854, 1.964285714285714e-05, 1.3214285714285712e-05]
fsky 0.75
CLASS default parameters {'output': 'tCl lCl mPk', 'non linear': 'halofit', 'l_max_scalars': 2000, 'lensing': 'yes', 'A_s': 2.083e-09, 'n_s': 0.9649, 'alpha_s': 0.0, 'h': 0.6736, 'N_ur': 2.0328, 'N_ncdm': 1, 'm_ncdm': 0.06, 'tau_reio': 0.0544, 'omega_b': 0.02237, 'omega_cdm': 0.12, 'Omega_k': 0.0, 'P_k_max_h/Mpc': 2.0, 'z_pk': '0.0,6'}


In [19]:
# DESI 2
zbin_edge = np.linspace(0.15, 2.3, 8)
zbin_center = 0.5 * (zbin_edge[1:] + zbin_edge[:-1])
print('zbin_center = ', zbin_center)

zbin_center =  [0.30357143 0.61071429 0.91785714 1.225      1.53214286 1.83928571
 2.14642857]


In [20]:
params = summary['CLASS default parameters']
cosmo = Class() 
cosmo.set(params) 
cosmo.compute() 

### (1b) The experiment

Now we specify the experiment, which is an instance of $\verb|experiment.py|$. At a minimum, we need to specify the redshift range of the survey ($z_\text{min}$ and $z_\text{max}$), the redshift binning, the sky coverage $f_\text{sky}$, the linear bias $b(z)$, and the number density $\bar{n}(z)$. The redshift binning can be specified in two ways: you can either input a `zedges` (numpy array) to specify the edges of the bins, or `nbins` (integer), in which case the redshift bins are assumed to be linearly spaced in $z$. 

In [21]:
# load fiducial linear bias/number density from table
zs,bs,ns = np.genfromtxt('/home/adrien/PDM/code/PDM2026_wsl/FishLSS_script/' + bfn + '.txt').T

# assume zs spans the full survey
ze = summary['Edges of redshift bins']
zmin = ze[0]
zmax = ze[-1]

z_centers = (np.array(ze[1:])+np.array(ze[:-1]))/2

# interpolate
b = interp1d(zs,bs)
n = interp1d(zs,ns)

nbins = len(ze)-1
fsky = summary['fsky']

We can now create the experiment:

In [22]:
exp = experiment(zmin=zmin, zmax=zmax, nbins=nbins, fsky=fsky, b=b, n=n)

In addition to the above, one can optionally specify the following in an `experiment` object.

- `b2` (function of $z$): quadratic bias $b_2(z)$ of the tracer, default $b_2 = 8(b-1)/21$
- `sigv` (float): the comoving velocity dispersion for FoG contributions [km/s], default is 100 km/s
- `sigma_z` (float): redshift error $\sigma_z/(1+z)$, assumed to be independent of redshift, default is 0

### (1c) The forecast object

With a cosmology and an experiment in hand, we can now create a forecast. Running the line below will create an `output` directory, as well as a subdirectory for the experiment of interest: `output/example_survey` in this case. After creating these directories, $\verb|FishLSS|$ will calculate the fiducial power spectra ($P_{gg}(\boldsymbol{k},z)$ and $P_\text{recon}(\boldsymbol{k},z)$) at the center of each redshift bin, and store them in `output/example_survey/derivatives/` and `output/example_survey/derivatives_recon` respectively (assuming that the files don't already exist). $\verb|FishLSS|$ will also calculate $C^{\kappa\kappa}_\ell$, $C^{\kappa g_i}_\ell$ and $C^{g_i g_i}_\ell$ for each redshift bin, and save them in `output/example_survey/derivatives_Cl`. 

In [23]:
# By default FishLSS won't recompute the fiducial power spectra (unless overwrite=True),
# instead, FishLSS will load them into memory. (assuming "python example_BAO_sigma8.py setup"
# has been run)
name = summary['Forecast name']
forecast = fisherForecast(experiment=exp,cosmo=cosmo,name=name,basedir=bd)

Redshift-space power spectra are computed on a (flattened) $k-\mu$ grid. That is, $P_{gg}(\boldsymbol{k},z)$ is stored as an array of length `forecast.Nk * forecast.Nmu`, with the corresponding values of $k$ and $\mu$ stored in `forecast.k` and `forecast.mu`. 

## (2) BAO forecast

As described in $\S3.6$ of [2106.09713](https://arxiv.org/pdf/2106.09713.pdf), we hold the shape of the fiducial power spectrum fixed in our BAO forecasts. We then find the errors on the two A-P parameters ($\alpha_\perp$, $\alpha_\parallel$) after marginalizing over the linear bias $b$ and 15 broad-band polynomials $\sum_{n=0}^4\sum_{m=0}^2 c_{nm}k^n\mu^{2m}$. We finally intepret the errors on the A-P parameters as the relative errors of $D_A(z)/r_d$ and $H(z)r_d$, where $r_d$ is sound horizon at the drag epoch.

Marginalizing over the polynomial coefficients is trivial to do analytically, so we only need to numerically compute derivatives with respect to $\alpha_\perp,\alpha_\parallel$ and $b$. 

In [24]:
basis = np.array(['alpha_perp','alpha_parallel','b'])

# set recon = True, so that we perform BAO reconstruction when computing the power spectrum
forecast.recon = True

# set the "marginalized parameters", aka the derivatives, to be [alpha's, linear b]
forecast.free_params = basis

In [25]:
# derivatives should have already been computed using "python example_BAO_sigma8.py example_survey rec"

# can also simply use:
# forecast.compute_derivatives()
# but this is slow in Jupyter

The derivatives have been automatically stored in `output/example_survey/derivatives_recon`. To load these derivatives, simply run:

In [26]:
derivs = forecast.load_derivatives(basis)

Now let's compute the Fisher matrices in each redshift bin using `get_fisher`, which takes the arguments:

- `basis` (np array): basis of the Fisher matrix 
- `globe` (int): let's not worry about this for now, it's value isn't important for computing the Fisher matrix for a single redshift bin
- `derivatives` (np array): load the derivatives from memory, if not specificied $\verb|FishLSS|$ will recalculate them (takes a lot of time!)
- `zbins` (np array): an array of ints specifying which redshift bins to include in the Fisher matrix. In this case we're computing a Fisher matrix for each redshift bin, so we set `zbins = np.array([i])` to get the Fisher matrix for the i'th bin.

In addition the the above you can also specify `kmax` or `kmax_knl` (ratio of $k_\text{max}$ to the non-linear scale at the center of each redshift bin). By default we set `kmax_knl=1` and `kmax=-10`. If `kmax` is set to be a positive number, then the code ignores `kmax_knl` and uses `kmax` instead.

In [27]:
# get the fisher matrices in each of the 3 redshift bins
F = lambda i: forecast.gen_fisher(basis, 100, derivatives=derivs, zbins=np.array([i]))
Fs = [F(i) for i in range(nbins)]
Fs = np.array(Fs)

Since we set `recon = True` when computing the Fisher matrices, $\verb|FishLSS|$ automatically knows to marginalize over the 15 polynomials, so each Fisher matrix will be an $18\times18$ matrix with basis $\{\alpha_\perp,\alpha_\parallel,b,c_{00},\cdots\}$

In [28]:
print(Fs.shape)

(7, 18, 18)


Now lets invert and compute the errors on the A-P parameters.

In [29]:
Finvs = [np.linalg.inv(Fs[i]) for i in range(nbins)]
saperp = [np.sqrt(Finvs[i][0,0]) for i in range(nbins)]
saparr = [np.sqrt(Finvs[i][1,1]) for i in range(nbins)]

From which we get the relative error on $D_A(z)/r_d$ and $H(z)r_d = r_d/D_H(z)$ in each redshift bin:

In [35]:
print('Relative error on DA/rd:',saperp)
print('Relative error on H*rd:',saparr)

Relative error on DA/rd: [np.float64(0.007770420705615622), np.float64(0.004222055118142484), np.float64(0.004130939073908244), np.float64(0.007821915980465841), np.float64(0.009697268024800754), np.float64(0.03421690981953927), np.float64(0.058216822003282216)]
Relative error on H*rd: [np.float64(0.012776527406970656), np.float64(0.006804041758996484), np.float64(0.006154188083637742), np.float64(0.009106251168603887), np.float64(0.011340308835418103), np.float64(0.03829243387404704), np.float64(0.06279284272478007)]


(For absolute errors : If $\sigma_\mathrm{parr}=$`saparr`, error on $D_H(z)/r_d$ is $\sigma_\mathrm{parr}/\alpha_\mathrm{parr}^2$)

But we have relative errors so it's the same

(Since $D_A(z) = D_M/(1+z)$ then if $a = D_A(z)/r_d$, we have $D_M(z)/r_d = a\cdot (1+z)$ and error : $\sigma_a \cdot (1+z)$)

In fact the rel. error on $D_M(z)/r_d$ is $\sqrt{(\sigma_a/a)^2 + (\sigma_z/(1+z))^2}$ which is then the same if we consider $\sigma_z = 0$

However since $z$ is taken for example in the middle of the bin there is actually an error on $z$ but it is no natural to include it here as $D_H/r_d$ hasn't a similar term in its error but has the same bins

In [21]:
a_perp = np.array(saperp) * (1 + z_centers)
print('Absolute error on DA:',a_perp)

Absolute error on DA: [0.01764059 0.01193556 0.0139778  0.0308216  0.04361003 0.17292805
 0.32663173]


### Create mean and cov file for cobaya

In [43]:
Cov_alpha = [np.zeros((2,2)) for _ in range(nbins)]
for i in range(nbins):
    Cov_alpha[i][0,0] = Finvs[i][0,0]
    Cov_alpha[i][1,1] = Finvs[i][1,1]
    Cov_alpha[i][0,1] = Finvs[i][0,1]
    Cov_alpha[i][1,0] = Finvs[i][1,0]

cov_mat = np.zeros((2*nbins, 2*nbins))
for i in range(nbins):
    cov_mat[2*i:2*i+2, 2*i:2*i+2] = Cov_alpha[i]

In [66]:
def create_DESI_fid_data(redshifts, DESI_style=False):
    cosmo_planck = cosmoprimo.fiducial.DESI()
    bkg = cosmo_planck.get_background(engine="class")
    thermo = cosmo_planck.get_thermodynamics()
    rdrag = thermo.rs_drag  # no parentheses needed, it's a property

    DM = bkg.comoving_angular_distance(redshifts)
    DH = 1 / bkg.efunc(redshifts) * 2997.92  # c/H(z) in Mpc, c=299792 km/s, H0 in km/s/Mpc
    DV = (redshifts * DM**2 * DH)**(1/3)
    DM_rd = DM / rdrag
    DH_rd = DH / rdrag
    DV_rd = DV / rdrag

    data_typ = ['DH_over_rs', 'DM_over_rs', 'DV_over_rs']

    fake_data = []
    if DESI_style:
        for i in range(1, len(redshifts)):
            line = [f"{redshifts[i]:.8e}", f"{DM_rd[i]:.8e}", data_typ[1]]
            fake_data.append(line)
            line = [f"{redshifts[i]:.8e}", f"{DH_rd[i]:.8e}", data_typ[0]]
            fake_data.append(line)

        line = [f"{float(redshifts[0]):.8e}", f"{DV_rd[0]:.8e}", data_typ[2]]
        fake_data.insert(0, line)
    else:
        for i in range(len(redshifts)):
            line = [f"{redshifts[i]:.8e}", f"{DM_rd[i]:.8e}", data_typ[1]]
            fake_data.append(line)
            line = [f"{redshifts[i]:.8e}", f"{DH_rd[i]:.8e}", data_typ[0]]
            fake_data.append(line)

    return np.array(fake_data)

def save_mean_data(fake_data, folder, filename, overwrite=False):
    if not os.path.exists(folder):
        os.makedirs(folder)
    f = folder + '/' + filename
    if os.path.exists(f):
        if overwrite:
            print("Overwriting file:", f)
            np.savetxt(f, fake_data, fmt="%s")
        else:
            print("File already exists:", f)
    else:
        np.savetxt(f, fake_data, fmt="%s")



def save_cov_from_Fisher(Fish_inv, nbins, folder, filename, DESI_style=False, overwrite=False):
    if DESI_style:
        print("A coder")
    else:
        cov_mat = np.zeros((2*nbins, 2*nbins))

        for i in range(nbins):
            cov_mat[2*i:2*i+2, 2*i:2*i+2] = Fish_inv[i][0:2, 0:2]
    
    if not os.path.exists(folder):
        os.makedirs(folder)
    f = folder + '/' + filename
    if os.path.exists(f):
        if overwrite:
            print("Overwriting file:", f)
            np.savetxt(f, cov_mat, fmt="%.8e")
        else:
            print("File already exists:", f)
    else:
        np.savetxt(f, cov_mat, fmt="%.8e")

    

In [67]:
# save_cov_from_Fisher(Finvs, nbins, 'fisher_forecast/'+bfn, bfn+'_cov.txt', overwrite=True)

In [ ]:
z = summary['Centers of redshift bins']
# save_mean_data(create_DESI_fid_data(z), 'fisher_forecast/'+bfn, bfn+'_mean.txt', overwrite=True)

Overwriting file: fisher_forecast/DESI2_fullsky/DESI2_fullsky_mean.txt


### Actual DESI errors :

#### definitions

In [24]:
def get_DESI_errors():
    dict_distance, z_DESI, path = get_DESI_data()

    cov_mat = np.loadtxt(path.replace('mean', 'cov'))

    errors = {}
    for i in range(len(cov_mat[0, :])):
        z = z_DESI[i//2]
        if i == 0:
            _, _, DVfid = get_DESI_fid(z)
            a_iso_err = np.sqrt(cov_mat[i][i]) / DVfid
            errors[z] = {'d_iso': a_iso_err}
        elif i%2 == 0:
            cov = [[cov_mat[i-1, i-1], cov_mat[i-1, i]], [cov_mat[i, i-1], cov_mat[i, i]]]
            sig_x = np.sqrt(cov[0][0])
            sig_y = np.sqrt(cov[1][1])

            DMfid, DHfid, DVfid = get_DESI_fid(z)
            if i == len(cov_mat[0, :])-1:
                d_a_perp = sig_y / DMfid
                d_a_par = sig_x / DHfid                
            else :
                d_a_perp = sig_x / DMfid
                d_a_par = sig_y / DHfid

            DM = dict_distance[z]['DM_over_rs']
            DH = dict_distance[z]['DH_over_rs']
            DV = (z*DM**2*DH)**(1/3)

            a_perp = DM / DMfid
            a_par = DH / DHfid
            a_iso = DV / DVfid
            a_AP = a_par / a_perp

            d_a_AP = a_AP * np.sqrt((d_a_par / a_par)**2 + (d_a_perp / a_perp)**2)
            d_a_iso = a_iso * np.sqrt((2/3)**3 * (d_a_perp / a_perp)**2 + (1/3)**2 * (d_a_par / a_par)**2)

            errors[z] = {'d_perp':  d_a_perp,
                         'd_par':   d_a_par,
                         'd_AP':    d_a_AP,
                         'd_iso':   d_a_iso
                         }
    return errors, z_DESI

def get_DESI_data():
    z_DESI = []
    path = '/home/adrien/PDM/code/desi_bao_dr2/desi_gaussian_bao_ALL_GCcomb_mean.txt'
    with open(path, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.split()
            z_DESI.append(float(parts[0]))

    z_DESI = set(z_DESI)
    z_DESI = list(z_DESI)
    z_DESI.sort()
    dict_distance = {}
    for z in z_DESI:
        dict_distance[z] = {}

    with open(path, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.split()
            z = float(parts[0])
            for key in dict_distance.keys():
                if key == z:
                    dict_distance[key][parts[2]] = float(parts[1])
    
    return dict_distance, z_DESI, path

def get_DESI_fid(redshift):
    DESI = cosmoprimo.fiducial.DESI(engine='class')
    DESI_bkg = DESI.get_background()
    DESI_thermo = DESI.get_thermodynamics()
    rdrag_fid = DESI_thermo.rs_drag
    DM_fid = DESI_bkg.comoving_angular_distance(redshift)
    DH_fid = 1 / DESI_bkg.efunc(redshift) * 2997.92458 # c/H(z) in Mpc, c=299792 km/s, H0 in km/s/Mpc
    DV_fid = (redshift * DM_fid**2 * DH_fid)**(1/3)
    DMover_rd_fid = DM_fid / rdrag_fid
    DHover_rd_fid = DH_fid / rdrag_fid
    DVover_rd_fid = DV_fid / rdrag_fid

    return DMover_rd_fid, DHover_rd_fid, DVover_rd_fid

#### DESI errors

In [26]:
true_aperp = []
true_aparr = []

errors, z_DESI = get_DESI_errors()

for z in z_DESI[1:]:
    true_aperp.append(errors[z]['d_perp'])
    true_aparr.append(errors[z]['d_par'])

In [27]:
print(true_aperp)
print(saperp[1:])

[np.float64(0.012471117519878885), np.float64(0.01016517787695462), np.float64(0.007355455756145305), np.float64(0.01155761710316807), np.float64(0.025220388963865248), np.float64(0.013569598490646746)]
[np.float64(0.007312813977064749), np.float64(0.007154996359312057), np.float64(0.013547955892363843), np.float64(0.016796160910827005), np.float64(0.05926542628741175), np.float64(0.10083449355260717)]


We're done with BAO forecasting, so let's set `recon = False`

In [ ]:
# forecast.recon = False